In [2]:
from typing import cast

import pandas as pd

from lib.types import FAQGroundTruthRecord


faq_ground_truth_df = pd.read_csv("data/faq_ground_truth.csv")

display(faq_ground_truth_df.head())

faq_ground_truth: list[FAQGroundTruthRecord] = cast(
    list[FAQGroundTruthRecord],
    faq_ground_truth_df.to_dict(orient="records"))

,question,document
0,I just found this course late — can I still st...,74eb249bbf
1,Is it too late to enroll if I discovered the c...,74eb249bbf
2,Can I still take part in the course even thoug...,74eb249bbf
3,"If I join the course late, am I still eligible...",74eb249bbf
4,What’s the deadline for submitting the project...,74eb249bbf


In [ ]:
from pydash import filter_

from lib.sources import load_faq_documents


predicate = {"course": "llm-zoomcamp"}

documents = filter_(load_faq_documents(), predicate)

In [ ]:
from lib.search import MinsearchLexicalSearchTool
from lib.search_types import LexicalSearchConfig


minsearch_lexical_search_tool = MinsearchLexicalSearchTool.from_documents(
    documents,
    text_fields=["question", "answer"],
    keyword_fields=["course"],
    config=LexicalSearchConfig(
        num_results=6,
        boost_dict={
            "question": 1.0,
            "answer": 3.0,
        }
    )
)

In [5]:
question = faq_ground_truth[0]["question"]

print(question)

print(minsearch_lexical_search_tool.search(question))

I just found this course late — can I still start it and join in now?
[{'id': '977bf7786c', 'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?', 'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."}, {'id': '74eb249bbf', 'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'I just discovered the course. Can I still join?', 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}, {'id': 'aa310de435', 'course': 'llm-zoomcamp', 'section': 'Module 1: RAG', 'question': 'Can I run the course locally instead of Codespaces?', 'answer': 'Yes. Codespac

In [6]:
from lib.types import FAQDocument, FAQGroundTruthRecord


def are_results_relevant(
        results: list[FAQDocument],
        ground_truth: FAQGroundTruthRecord) -> list[int]:
    return [int(result["id"] == ground_truth["document"]) for result in results]

In [7]:
results = cast(
    list[FAQDocument],
    minsearch_lexical_search_tool.search(faq_ground_truth[0]["question"]))

print(results)

print(faq_ground_truth[0])

are_results_relevant(results, faq_ground_truth[0])

[{'id': '977bf7786c', 'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?', 'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."}, {'id': '74eb249bbf', 'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'I just discovered the course. Can I still join?', 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}, {'id': 'aa310de435', 'course': 'llm-zoomcamp', 'section': 'Module 1: RAG', 'question': 'Can I run the course locally instead of Codespaces?', 'answer': 'Yes. Codespaces is just the easiest way for everyone to start with the same environ

[0, 1, 0, 0, 0, 0]

In [8]:
from lib.search import SearchTool


def compute_total_relevance(
        faq_ground_truth: list[FAQGroundTruthRecord],
        search_tool: SearchTool[FAQDocument]
        ) -> list[list[int]]:
    total_relevance_matrix: list[list[int]] = []
    for ground_truth_record in faq_ground_truth:
        results = search_tool.search(ground_truth_record["question"])
        total_relevance_matrix.append(
            are_results_relevant(results, ground_truth_record))
    return total_relevance_matrix


relevances_with_lexical_search = compute_total_relevance(
    faq_ground_truth, minsearch_lexical_search_tool)

In [9]:
from scripts.embedder import Embedder

# encoder = SentenceTransformer("all-MiniLM-L6-v2")
encoder = Embedder("models/Xenova/all-MiniLM-L6-v2")

In [10]:
from lib.search import MinsearchSemanticSearchTool
from lib.search_types import SemanticSearchConfig


minsearch_semantic_search_tool = MinsearchSemanticSearchTool.from_documents(
    documents,
    encoder=encoder,
    text_fields=["question", "answer"],
    keyword_fields=["course"],
    config=SemanticSearchConfig(
        num_results=6,
    )
)

  0%|          | 0/2 [00:00<?, ?it/s]

In [11]:
relevances_with_semantic_search = compute_total_relevance(
    faq_ground_truth, minsearch_semantic_search_tool)

In [12]:
def compute_hit_rate(relevances: list[list[int]]) -> float:
    relevance_per_question = [any(relevance) for relevance in relevances]
    hit_rate = sum(relevance_per_question) / len(relevances)
    return hit_rate

def compute_mrr(relevances: list[list[int]]) -> float:
    total_score = 0.0
    for relevance in relevances:
        for rank in range(len(relevance)):
            if relevance[rank] == 1:
                score = 1 / (rank + 1)
                total_score += score
                break
    return total_score / len(relevances)

In [13]:
print(compute_hit_rate(relevances_with_lexical_search))
print(compute_hit_rate(relevances_with_semantic_search))
print(compute_mrr(relevances_with_lexical_search))
print(compute_mrr(relevances_with_semantic_search))

0.9858823529411764
0.9811764705882353
0.8961960784313723
0.9118431372549021


In [14]:
from pydash import find

print(relevances_with_semantic_search[-1])

print(faq_ground_truth[-1])

predicate = {"id": faq_ground_truth[-1]["document"]}

print(find(documents, predicate))

results = cast(
    list[FAQDocument],
    minsearch_semantic_search_tool.search(faq_ground_truth[-1]["question"])
)

print(are_results_relevant(results, faq_ground_truth[-1]))

[1, 0, 0, 0, 0, 0]
{'question': 'Is there a workaround for the requests version issue when using lancedb in this dlt session?', 'document': '4b30b918bc'}
{'id': '4b30b918bc', 'course': 'llm-zoomcamp', 'section': 'Workshop: Open-Source Data Ingestion (dlt)', 'question': 'Error: How to fix requests library only installs v2.28 instead of v2.32 required for lancedb?', 'answer': 'If you encounter a 401 Client Error, it may indicate the need to grant access to the key or that the key is incorrect.\n\nTo install the correct version directly from the source, use the following command:\n\n```bash\npip install "requests @ https://github.com/psf/requests/archive/refs/tags/v2.32.3.zip"\n```'}
[1, 0, 0, 0, 0, 0]


In [15]:
def evaluate(
        faq_ground_truth: list[FAQGroundTruthRecord],
        search_function: SearchTool[FAQDocument]):
    
    total_relevance = compute_total_relevance(faq_ground_truth, search_function)

    return {
        "hit_rate": compute_hit_rate(total_relevance),
        "mrr": compute_mrr(total_relevance),
    }

In [16]:
print(evaluate(faq_ground_truth, minsearch_lexical_search_tool))

print(evaluate(faq_ground_truth, minsearch_semantic_search_tool))

{'hit_rate': 0.9858823529411764, 'mrr': 0.8961960784313723}
{'hit_rate': 0.9811764705882353, 'mrr': 0.9118431372549021}


In [ ]:
from itertools import product


question_boost_dict_values_to_test = [1.0]
answer_boost_dict_values_to_test = [1.0, 2.0, 3.0, 5.0]

boost_dicts = [
    {
        "question": question_boost,
        "answer": answer_boost
    }
    for question_boost, answer_boost in product(
        question_boost_dict_values_to_test,
        answer_boost_dict_values_to_test
    )
]

print(boost_dicts)

[{'question': 1, 'answer': 1}, {'question': 1, 'answer': 2}, {'question': 1, 'answer': 3}, {'question': 1, 'answer': 5}]


In [18]:
from tqdm.auto import tqdm


class BoostDictEvaluationResults(TypedDict):
    boost_dict: dict[str, float]
    evaluation_results: dict[str, float]


evaluation_results_for_boost_dicts: list[BoostDictEvaluationResults] = []

for boost_dict in tqdm(boost_dicts):

    minsearch_lexical_search_tool = MinsearchLexicalSearchTool.from_documents(
        documents,
        text_fields=["question", "answer"],
        keyword_fields=["course"],
        config=LexicalSearchConfig(
            num_results=5,
            boost_dict=boost_dict
        )
    )

    evaluation_results = evaluate(faq_ground_truth, minsearch_lexical_search_tool)

    evaluation_results_for_boost_dicts.append({
        "boost_dict": boost_dict,
        "evaluation_results": evaluation_results
    })

evaluation_results_for_boost_dicts = sorted(
    evaluation_results_for_boost_dicts,
    key=lambda evaluation_result: (
        evaluation_result["evaluation_results"]["hit_rate"],
        evaluation_result["evaluation_results"]["mrr"],
    ),
    reverse=True,
)

  0%|          | 0/4 [00:00<?, ?it/s]

In [19]:
for _ in evaluation_results_for_boost_dicts[:15]:
    print(_)

{'boost_dict': {'question': 1, 'answer': 3}, 'evaluation_results': {'hit_rate': 0.9858823529411764, 'mrr': 0.8961960784313723}}
{'boost_dict': {'question': 1, 'answer': 2}, 'evaluation_results': {'hit_rate': 0.9764705882352941, 'mrr': 0.8889411764705879}}
{'boost_dict': {'question': 1, 'answer': 5}, 'evaluation_results': {'hit_rate': 0.971764705882353, 'mrr': 0.8834117647058823}}
{'boost_dict': {'question': 1, 'answer': 1}, 'evaluation_results': {'hit_rate': 0.9482352941176471, 'mrr': 0.848627450980392}}
